In [1]:
import torch
import torch.nn as nn

In [15]:
!pip show torch

Name: torch
Version: 2.0.1
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: C:\Users\Tharunraj_S\AppData\Roaming\Python\Python311\site-packages
Requires: filelock, jinja2, networkx, sympy, typing-extensions
Required-by: easyocr, thop, torchvision, ultralytics


In [18]:
import torch

class KidneyCNN(nn.Module):
    def __init__(self, input_size=24, num_classes=2):
        super(KidneyCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * (input_size - 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def __getstate__(self):
        state = self.state_dict()
        state['quantized_weights'] = self.qconfig_dict
        return state

    def __setstate__(self, state):
        self.load_state_dict(state)
        self.qconfig_dict = state['quantized_weights']

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Load the trained model
model = KidneyCNN()
model_path = r"C:\Users\Tharunraj_S\Desktop\miniproject\runs_epoch100\trained_model_epoch100.pt"
model.load_state_dict(torch.load(model))

# Quantize the model
traced_model = torch.jit.trace_module(model, {'forward': torch.randn(1, 24)})
quantized_model = torch.quantization.quantize_dynamic(traced_model, {'data': torch.randn(1, 24)})

# Save the quantized model
torch.jit.save(quantized_model, 'quantized_model.pt')


AttributeError: 'KidneyCNN' object has no attribute 'seek'. You can only torch.load from a file that is seekable. Please pre-load the data into a buffer like io.BytesIO and try to load from it instead.